In [11]:
import pandas as pd
from sklearn.neighbors import BallTree
import numpy as np
import time
import requests
import unicodedata

In [12]:
hdb_final = pd.read_csv("../data/cleaned/hdb_final.csv")
sch_data = pd.read_csv("../outputs/Good_School_index.csv")

In [13]:
# lat lon for sch

SEARCH_URL = "https://www.onemap.gov.sg/api/common/elastic/search"

def geocode_onemap(search_val, session=None, auth_token=None, retries=3, sleep=0.3):
    s = session or requests.Session()
    headers = {"Authorization": auth_token} if auth_token else {}
    params = {
        "searchVal": search_val,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }

    for i in range(retries):
        try:
            r = s.get(SEARCH_URL, params=params, headers=headers, timeout=15)
            r.raise_for_status()
            j = r.json()
            if int(j.get("found", 0)) > 0:
                best = j["results"][0]
                return float(best["LATITUDE"]), float(best["LONGITUDE"])
            return None, None
        except Exception:
            if i == retries - 1:
                return None, None
            time.sleep((i + 1) * sleep)



In [14]:
sess = requests.Session()

def normalize_name(s):
    if pd.isna(s):
        return s
    s = str(s).strip()
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("’", "'").replace("‘", "'")
    return s

sch_data["search_school"] = sch_data["school"].map(normalize_name)

school_overrides = {
    "CHIJ (Toa Payoh)": "CHIJ Primary (Toa Payoh)",
    "St. Andrew's Junior": "St. Andrew's Junior School",
    "St. Anthony's": "St. Anthony's Primary School",
    "St. Anthony's Canossian": "St. Anthony's Canossian Primary School",
    "St. Gabriel's": "St. Gabriel's Primary School",
    "St. Hilda's": "St. Hilda's Primary School",
    "St. Joseph's Institution Junior": "St. Joseph's Institution Junior",
    "St. Margaret's": "St. Margaret's Primary School",
    "St. Stephen's": "St. Stephen's School",
    "CHIJ (Katong)": "CHIJ Katong Primary",
    "CHIJ Our Lady of Good Counsel": "CHIJ Our Lady of Good Counsel",
    "CHIJ Our Lady Queen of Peace": "CHIJ Our Lady Queen of Peace",
    "CHIJ (Kellock)": "CHIJ Kellock",
}

def geocode_school(name, session=None):
    # manual override first for known failures
    if name in school_overrides:
        queries = [school_overrides[name]]
    else:
        # generic rule for all others
        queries = [
            f"{name} Primary School",
            f"{name} School"
        ]
    
    for q in queries:
        lat, lon = geocode_onemap(q, session=session)
        if lat is not None and lon is not None:
            return lat, lon, q
    
    return None, None, None


sch_unique = sch_data["search_school"].dropna().unique()

sch_map = {}
sch_query_used = {}

for s in sch_unique:
    lat, lon, q_used = geocode_school(s, session=sess)
    sch_map[s] = (lat, lon)
    sch_query_used[s] = q_used

sch_data["lat"] = sch_data["search_school"].map(lambda x: sch_map.get(x, (None, None))[0])
sch_data["lon"] = sch_data["search_school"].map(lambda x: sch_map.get(x, (None, None))[1])
sch_data["query_used"] = sch_data["search_school"].map(sch_query_used)

In [15]:
# check for NaN lat/lon 
print("Schools with missing lat/lon:")
print(sch_data[sch_data["lat"].isna() | sch_data["lon"].isna()][["school","search_school", "lat", "lon"]])

Schools with missing lat/lon:
Empty DataFrame
Columns: [school, search_school, lat, lon]
Index: []


In [16]:
merged_df = hdb_final.copy()
sch_df = sch_data.copy()

merged_df["year"] = pd.to_datetime(merged_df["month"]).dt.year
sch_df["year"] = sch_df["year"].astype(int)

# keep rows with valid coordinates
merged_df = merged_df.dropna(subset=["lat", "lon"]).copy()
sch_df = sch_df.dropna(subset=["lat", "lon"]).copy()

good_cols = ["good_school_75", "good_school_80", "good_school_85", "good_school_90"]

for col in good_cols:
    sch_df[col] = sch_df[col].fillna(0).astype(int)

# initialize output columns
merged_df["Countall_<1"] = 0
merged_df["Countall_1-2"] = 0

for col in good_cols:
    merged_df[f"Count{col}_<1"] = 0
    merged_df[f"Count{col}_1-2"] = 0
    merged_df[f"D_1_{col}"] = 0
    merged_df[f"D_1-2_{col}"] = 0

earth_radius_km = 6371

# process year by year
for yr in sorted(merged_df["year"].dropna().unique()):
    house_idx = merged_df.index[merged_df["year"] == yr]
    school_y = sch_df[sch_df["year"] == yr].copy()

    # if no schools for that year, leave zeros
    if school_y.empty:
        continue

    house_y = merged_df.loc[house_idx]

    # convert lat/lon to radians
    house_coords = np.radians(house_y[["lat", "lon"]].to_numpy())
    school_coords = np.radians(school_y[["lat", "lon"]].to_numpy())

    # build BallTree for same-year schools only
    tree = BallTree(school_coords, metric="haversine")

    # neighbours within 1km and 2km
    ind_1km = tree.query_radius(house_coords, r=1 / earth_radius_km)
    ind_2km = tree.query_radius(house_coords, r=2 / earth_radius_km)

    # ALL SCHOOL COUNTS
    countall_1km = np.array([len(i) for i in ind_1km])
    countall_2km = np.array([len(i) for i in ind_2km])

    merged_df.loc[house_idx, "Countall_<1"] = countall_1km
    merged_df.loc[house_idx, "Countall_1-2"] = countall_2km - countall_1km

    # GOOD SCHOOL FEATURES
    for col in good_cols:
        school_vals = school_y[col].to_numpy()

        countgood_1km = np.array([school_vals[idx].sum() for idx in ind_1km])
        countgood_2km = np.array([school_vals[idx].sum() for idx in ind_2km])

        merged_df.loc[house_idx, f"Count{col}_<1"] = countgood_1km
        merged_df.loc[house_idx, f"Count{col}_1-2"] = countgood_2km - countgood_1km

        merged_df.loc[house_idx, f"D_1_{col}"] = (countgood_1km > 0).astype(int)
        merged_df.loc[house_idx, f"D_1-2_{col}"] = ((countgood_2km - countgood_1km) > 0).astype(int)

merged_df.head()

,month,town,flat_type,block,street_name,floor_area_sqm,flat_model,lease_commence_date,resale_price,year,...,D_1_good_school_80,D_1-2_good_school_80,Countgood_school_85_<1,Countgood_school_85_1-2,D_1_good_school_85,D_1-2_good_school_85,Countgood_school_90_<1,Countgood_school_90_1-2,D_1_good_school_90,D_1-2_good_school_90
0,2009-01-01,ANG MO KIO,2,314,ANG MO KIO AVE 3,44.0,Improved,1978,188000.0,2009,...,0,0,0,0,0,0,0,0,0,0
1,2009-01-01,ANG MO KIO,3,174,ANG MO KIO AVE 4,60.0,Improved,1986,182000.0,2009,...,0,0,0,0,0,0,0,0,0,0
2,2009-01-01,ANG MO KIO,3,215,ANG MO KIO AVE 1,73.0,New Generation,1976,278000.0,2009,...,0,0,0,0,0,0,0,0,0,0
3,2009-01-01,ANG MO KIO,3,219,ANG MO KIO AVE 1,82.0,New Generation,1977,298000.0,2009,...,0,0,0,0,0,0,0,0,0,0
4,2009-01-01,ANG MO KIO,3,219,ANG MO KIO AVE 1,67.0,New Generation,1977,238000.0,2009,...,0,0,0,0,0,0,0,0,0,0


In [17]:
# drop 2009 - 2012 since no school data (used for rolling window)
merged_df = merged_df[merged_df["year"] >= 2013].copy()


In [18]:
merged_df.head()

,month,town,flat_type,block,street_name,floor_area_sqm,flat_model,lease_commence_date,resale_price,year,...,D_1_good_school_80,D_1-2_good_school_80,Countgood_school_85_<1,Countgood_school_85_1-2,D_1_good_school_85,D_1-2_good_school_85,Countgood_school_90_<1,Countgood_school_90_1-2,D_1_good_school_90,D_1-2_good_school_90
110545,2013-01-01,ANG MO KIO,2,510,ANG MO KIO AVE 8,44.0,Improved,1980,253000.0,2013,...,0,1,0,1,0,1,0,1,0,1
110546,2013-01-01,ANG MO KIO,2,314,ANG MO KIO AVE 3,44.0,Improved,1978,270000.0,2013,...,0,1,0,3,0,1,0,3,0,1
110547,2013-01-01,ANG MO KIO,2,323,ANG MO KIO AVE 3,44.0,Improved,1977,283000.0,2013,...,0,1,0,3,0,1,0,3,0,1
110548,2013-01-01,ANG MO KIO,3,170,ANG MO KIO AVE 4,61.0,Improved,1986,305000.0,2013,...,1,1,1,1,1,1,1,1,1,1
110549,2013-01-01,ANG MO KIO,3,174,ANG MO KIO AVE 4,60.0,Improved,1986,320000.0,2013,...,1,1,1,1,1,1,1,1,1,1


In [19]:
# rename cols
merged_df = merged_df.rename(columns={
    "Countall_<1": "countall_0_1km",
    "Countall_1-2": "countall_1_2km",
    "Countgood_school_75_<1": "count_0_1km_good75",
    "Countgood_school_75_1-2": "count_1_2km_good75",
    "Countgood_school_80_<1": "count_0_1km_good80",
    "Countgood_school_80_1-2": "count_1_2km_good80",
    "Countgood_school_85_<1": "count_0_1km_good85",
    "Countgood_school_85_1-2": "count_1_2km_good85",
    "Countgood_school_90_<1": "count_0_1km_good90",
    "Countgood_school_90_1-2": "count_1_2km_good90",
    "D_1_good_school_75": "d_0_1km_good75",
    "D_1-2_good_school_75": "d_1_2km_good75",
    "D_1_good_school_80": "d_0_1km_good80",
    "D_1-2_good_school_80": "d_1_2km_good80",
    "D_1_good_school_85": "d_0_1km_good85",
    "D_1-2_good_school_85": "d_1_2km_good85",
    "D_1_good_school_90": "d_0_1km_good90",
    "D_1-2_good_school_90": "d_1_2km_good90"
})

merged_df.head()

,month,town,flat_type,block,street_name,floor_area_sqm,flat_model,lease_commence_date,resale_price,year,...,d_0_1km_good80,d_1_2km_good80,count_0_1km_good85,count_1_2km_good85,d_0_1km_good85,d_1_2km_good85,count_0_1km_good90,count_1_2km_good90,d_0_1km_good90,d_1_2km_good90
110545,2013-01-01,ANG MO KIO,2,510,ANG MO KIO AVE 8,44.0,Improved,1980,253000.0,2013,...,0,1,0,1,0,1,0,1,0,1
110546,2013-01-01,ANG MO KIO,2,314,ANG MO KIO AVE 3,44.0,Improved,1978,270000.0,2013,...,0,1,0,3,0,1,0,3,0,1
110547,2013-01-01,ANG MO KIO,2,323,ANG MO KIO AVE 3,44.0,Improved,1977,283000.0,2013,...,0,1,0,3,0,1,0,3,0,1
110548,2013-01-01,ANG MO KIO,3,170,ANG MO KIO AVE 4,61.0,Improved,1986,305000.0,2013,...,1,1,1,1,1,1,1,1,1,1
110549,2013-01-01,ANG MO KIO,3,174,ANG MO KIO AVE 4,60.0,Improved,1986,320000.0,2013,...,1,1,1,1,1,1,1,1,1,1


In [20]:
# save as csv
merged_df.to_csv("../data/final_df.csv", index=False)